<a href="https://colab.research.google.com/github/cbishop4/MSE7530/blob/main/ClassNotebooks/F2026/Lecture12_InClass_20261014.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Hands-on AFM analysis - October 14th, 2026
This notebook was started from the "completedinclass" file that was posted last week. I have left some of that material intact; we will run through it very fast just to ensure we have all of our variables.

In [ ]:
!pip install uv -q
!uv pip install pySPM --system -q

In [ ]:
import pySPM as spm
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from matplotlib.pyplot import cm

In [ ]:
# Download the AFM data from the public repo
!git clone https://github.com/cbishop4/Alahe2026.git

In [ ]:
import re

In [ ]:
rawAFM = '/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030'
scan = spm.Bruker(rawAFM)
ht_chan = scan.get_channel('Height')
ht_px_vals = ht_chan.pixels
phz_chan = scan.get_channel('Phase')
phz_px_vals = phz_chan.pixels

### <font color='navy'>Function to read scan size in nm (completed 10/7)</font>

In [ ]:
def pull_params(filename,print_explicit=False):
  scan = spm.Bruker(filename) # method to load a Bruker file
  scan_size = scan.layers[0][b'Scan Size'] # returns raw string
  num_lines = scan.layers[0][b'Number of lines'] # returns raw string
  ss = float(re.findall(r'\d+(?:\.\d+)?', str(scan_size[0]))[0]) # uses regex to pull out the float from the scan size string
  nlines = int(re.findall(r'\d+', str(num_lines[0]))[0]) # uses regex to pull out the integer from the "number of lines" string
  if ss < 45.: # scan size in metadata is stored in microns
    ss_nm = ss*1000. # convert microns to nm
  else:
    ss_nm = ss # number is large enough that we assume it was recorded in nm
  px_nm = ss_nm/nlines # divide scan size in nm by the number of lines (resolution)
  if print_explicit: # if we change it to True
    print(f'The scan size is {ss_nm} nm, there are {nlines} lines, and the pixel size is {px_nm} nm')
  return ss_nm, nlines, px_nm

In [ ]:
#testing
pull_params('/content/Alahe2026/AFMdata/0p04nms/MAH25_01.024')

# Task 2. Roughness quantification. Can be done with any numpy array!

Together on the board, we worked through the mathematical definitions of the five measures of roughness that are schematically illustrated in the slides. <font color='blue'>Then, below, we wrote cells that calculate these for a 2D Numpy Array `R_q`, `R_a`, `R_p`, `R_v`, and `R_T` so that the print cell below works.</font>

### <font color='magenta'>Answers for the stuff I asked you to do on your own</font>

In [ ]:
heights = ht_px_vals
mean_ht = np.mean(heights)

In [ ]:
R_q = np.sqrt((1/heights.size)*np.sum((heights-mean_ht)**2)) #rms roughness

In [ ]:
R_a = (1/heights.size)*np.sum(np.abs(heights - mean_ht)) # average roughness

In [ ]:
R_p = np.max(heights) - mean_ht # maximum peak height from avg

In [ ]:
R_v = np.abs(np.min(heights) - mean_ht) # maximum valley depth from avg

In [ ]:
R_T = np.max(heights) - np.min(heights) # total range

In [ ]:
print(f'Rq: {R_q} nm')
print(f'Ra: {R_a} nm')
print(f'Rp: {R_p} nm')
print(f'Rv: {R_v} nm')
print(f'RT: {R_T} nm')

<font color='blue'>Now turn these into user-defined functions. You can write a separate function for each, or a single function that returns all five (i.e., `return R_q, R_a, R_p, R_v, R_T`). The loops later in the notebook use a single function called `calc_Rmeasures(heights)`, so use that name for the combined version.</font>

In [ ]:
def calc_Rmeasures(heights):
  mean_ht = np.mean(heights)
  R_q = np.sqrt((1/heights.size)*np.sum((heights-mean_ht)**2)) #rms roughness
  R_a = (1/heights.size)*np.sum(np.abs(heights - mean_ht)) # average roughness
  R_p = np.max(heights) - mean_ht # maximum peak height from avg
  R_v = np.abs(np.min(heights) - mean_ht) # maximum valley depth from avg
  R_T = np.max(heights) - np.min(heights) # total range
  return R_q, R_a, R_p, R_v, R_T

<font color='magenta'><font size="6"> Task 3 - you should have tried this, we will go through together briefly.</font></font>

# Task 3. Compare across processing conditions (deposition rates).
Now we will do something analogous to what you'll be doing in the project: comparing some parameters to the processing condition. I'll start by listing a few things that will help you move through the data automatically. We'll go through these together and take notes on what exactly is happening.

In [ ]:
# deposition rate labels (used in folder names) and the same rates as numbers, in nm/s
rate_codes = ['0p8','0p4','0p08','0p04','0p008']
rs_floats = np.asarray([0.8, 0.4, 0.08, 0.04, 0.008])

# one representative raw AFM file for each deposition rate
filelist = ['/content/Alahe2026/AFMdata/0p8nms/MAH27.037',
            '/content/Alahe2026/AFMdata/0p4nms/MAH23_01.012',
            '/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030',
            '/content/Alahe2026/AFMdata/0p04nms/MAH25_01.024',
            '/content/Alahe2026/AFMdata/0p008nms/MAH29.047']

First, let's just plot all of these.

In [ ]:
channel_name = 'Height'
fig, ax = plt.subplots(1,5,figsize=(20,3))
for i,f in enumerate(filelist):
  scan = spm.Bruker(f)
  channel1 = scan.get_channel(channel_name)
  channel_corr = channel1.correct_lines(inline=False)
  channel = channel_corr.pixels
  ax[i].imshow(channel)
  ax[i].set_title(f'R_dep = {rs_floats[i]} nm/s')

N.B. I decided to add in some code for scale bars here.

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

In [ ]:
channel_name = 'Height'
fig, ax = plt.subplots(1,5,figsize=(15,3))
for i,f in enumerate(filelist):
  scan = spm.Bruker(f)
  channel1 = scan.get_channel(channel_name)
  channel_corr = channel1.correct_lines(inline=False)
  channel = channel_corr.pixels
  ss_nm, nlines, px_nm = pull_params(f,print_explicit=False)
  cb = ax[i].imshow(channel,cmap='magma') # I changed the colormap for fun
  ax[i].set_title(f'R_dep = {rs_floats[i]} nm/s')
  fig.colorbar(cb, ax=ax[i], location='bottom',label='Height (nm)')
  scalebar = AnchoredSizeBar(ax[i].transData,
                           round(500 / px_nm), '500 nm', 'lower center',
                           pad=0.1,
                           color='white',
                           frameon=False,
                           size_vertical=3)

  ax[i].add_artist(scalebar)

These are all 2000 x 2000 nm images. For each file, calculate the five roughness measures and store them in the lists below; then run the plotting cell that follows.

In [ ]:
R_qs = []; R_as = []; R_ps = []; R_vs = []; R_Ts = []
for f in filelist:
  scan = spm.Bruker(f)
  channel1 = scan.get_channel(channel_name)
  channel_corr = channel1.correct_lines(inline=False)
  channel = channel_corr.pixels
  R_q, R_a, R_p, R_v, R_T = calc_Rmeasures(channel)
  R_qs.append(R_q)
  R_as.append(R_a)
  R_ps.append(R_p)
  R_vs.append(R_v)
  R_Ts.append(R_T)

In [ ]:
fig, ax = plt.subplots(1,5,figsize=(15,3))
plt.subplots_adjust(wspace=0.3)
ax[0].plot(rs_floats, R_qs,'-o')
ax[1].plot(rs_floats, R_as,'-o')
ax[2].plot(rs_floats, R_ps,'-o')
ax[3].plot(rs_floats, R_vs,'-o')
ax[4].plot(rs_floats, R_Ts,'-o')

for a in ax:
  a.semilogx()
  a.set_xlabel('$R_{dep}$ (nm/s)')
ax[0].set_ylabel('Rq (nm)')
ax[1].set_ylabel('Ra (nm)')
ax[2].set_ylabel('Rp (nm)')
ax[3].set_ylabel('Rv (nm)')
ax[4].set_ylabel('RT (nm)')

# Task 4. Analysis in reciprocal space.
Now we will transform the real-space AFM data to a reciprocal-space power spectrum.

In [ ]:
fname = '/content/Alahe2026/AFMdata/0p8nms/MAH27.037'
scan = spm.Bruker(fname)
channel1 = scan.get_channel('Height')
channel_corr = channel1.correct_lines(inline=False)
channel = channel_corr.pixels

In [ ]:
from scipy import stats

To look at what's happening here, I'm going to break down the steps a bit. This is **not** essential for you to know, but I'm hoping it explains it some. The top of the cell is code we will keep to do the operations later, the bottom is code to make plots to illustrate.
<font color='red'>One thing to be cautious about: this procedure does re-assign variables. You therefore **must** execute the cells in order, and only once each. After breaking down the steps, I will provide code that avoids this problem.</font>

In [ ]:
# pull the parameters needed to dimensionalize the result
ss_nm, npix, physsize_nm = pull_params(fname,print_explicit=False)
# take an n-dimensional fourier transform - here it is 2 dimensions, because you have a 2-dimensional image
fourier_image = np.fft.fftn(channel)
# |F(Q)|^2
fourier_amplitudes = np.abs(fourier_image)**2

# below here is just to illustrate what is happening
print(f'The Fourier image is a {fourier_image.shape} {type(fourier_image)} of {type(fourier_image[0,0])} (complex numbers)')
print(f'The Fourier amplitudes are a {fourier_amplitudes.shape} {type(fourier_amplitudes)} of {type(fourier_amplitudes[0,0])} (floats)')
print('They are plotted here.')
fig, ax = plt.subplots(1,3,figsize=(10,4))
cb = ax[0].imshow(fourier_image.real,norm=LogNorm(1e-3,1e5))
plt.colorbar(cb, ax=ax[0],location='bottom')
cb = ax[1].imshow(fourier_image.imag,norm=LogNorm(1e-3,1e5))
plt.colorbar(cb, ax=ax[1],location='bottom')
cb = ax[2].imshow(fourier_amplitudes,norm=LogNorm(1e-3,1e5))
plt.colorbar(cb, ax=ax[2],location='bottom')
ax[0].set_title('Real components of \n the Fourier transform')
ax[1].set_title('Imaginary components of \n the Fourier transform')
ax[2].set_title('Fourier amplitudes')

Now we're going to set up the k-values (analogous to q-values - this is just often the terminology used for power spectrums)

In [ ]:
# take the pixel dimensions and return the frequency bin centers in pixels^-1
kfreq = np.fft.fftfreq(npix)*npix
# makes a grid so that these axes are in both dimensions
kfreq2D = np.meshgrid(kfreq, kfreq)
# let's see
knrm = np.sqrt(kfreq2D[0]**2 + kfreq2D[1]**2)



# illustration code
fig, ax = plt.subplots(1,4,figsize=(15,4))
print(f'kfreq is a 1D {kfreq.shape}-element array constructed from {npix} pixels ')
ax[0].plot(kfreq)
ax[0].set_xlabel('Index'); ax[0].set_ylabel('Bin center ($nm^{-1}$)')
print(f'kfreq2D is a tuple containing two 2D grids with axes of kfreq')
cb = ax[1].imshow(kfreq2D[0])
plt.colorbar(cb, ax=ax[1],location='bottom',label='value ($pix^{-1}$)')
cb = ax[2].imshow(kfreq2D[1])
plt.colorbar(cb, ax=ax[2],location='bottom',label='value ($pix^{-1}$)')
#ax[1].set_xlabel('Index'); ax[1].set_ylabel('Bin center ($nm^{-1}$)')
cb = ax[3].imshow(knrm)
plt.colorbar(cb, ax=ax[3],location='bottom',label='value ($pix^{-1}$)')
for a in [ax[1],ax[2],ax[3]]:
  a.set_xlabel('Index'); a.set_ylabel('Index')
ax[0].set_title('kfreq')
ax[1].set_title('kfreq2D[0]')
ax[2].set_title('kfreq2D[1]')
ax[3].set_title('knrm')

The relevant things to pull out here are the 2D grid of k-values ($pix^{-1}$) and the Fourier amplitudes:

In [ ]:
fig, ax = plt.subplots(1,2)
plt.subplots_adjust(wspace=0.4)
cb = ax[0].imshow(knrm)
plt.colorbar(cb, ax=ax[0],location='bottom',label='Value ($pix^{-1}$)')
ax[0].set_title('knrm')

cb = ax[1].imshow(fourier_amplitudes,norm=LogNorm(1e-3,1e5))
plt.colorbar(cb, ax=ax[1],location='bottom',label='Value ($|F(k)|^{2}$)')
ax[1].set_title('Fourier amplitudes')

for a in ax:
  a.set_xlabel('Index'); a.set_ylabel('index')

They are then flattened into 1D arrays so that they can be further manipulated. These are large arrays:

In [ ]:
knrm = knrm.flatten()
fourier_amplitudes = fourier_amplitudes.flatten()
print(f'They are {len(knrm)}-element arrays (1024x1024)')

Now, we basically do a histogram: what is the mean intensity at each inverse pixel value?

In [ ]:
# generates "bins" in which to store the corresponding intensity values
kbins = np.arange(0.5, npix//2+1, 1.) # evenly spaced numbers from 0.5 to the number of pixels+0.5, in steps of 1
kvals = 0.5 * (kbins[1:] + kbins[:-1]) # goes from 1.0 to 513.0 (2 pixels per bin)

# next, we calculate the mean intensity in each bin (a histogram)
Abins, _, _ = stats.binned_statistic(knrm, fourier_amplitudes,
                                      statistic = "mean",
                                      bins = kbins)

plt.plot(kvals, Abins)
plt.loglog()
plt.xlabel('Bins ($pix^{-1}$)')
plt.ylabel('Mean intensity ($|F(k)|^{2}$')

Finally, we convert the axes into real units ($pix^{-1}$ to $nm^{-1}$)

In [ ]:
Abins *= np.pi * (kbins[1:]**2 - kbins[:-1]**2)
kvals = 2*np.pi*kvals/ ss_nm

plt.plot(kvals, Abins)
plt.loglog()
plt.xlabel('k ($nm^{-1}$)')
plt.ylabel('Mean intensity ($|F(k)|^{2}$')

### Code all together

In [ ]:
ss_nm, npix, physsize_nm = pull_params(fname,print_explicit=False)
fourier_image = np.fft.fftn(channel)
fourier_amplitudes = np.abs(fourier_image)**2
kfreq = np.fft.fftfreq(npix)*npix
kfreq2D = np.meshgrid(kfreq, kfreq)
knrm = np.sqrt(kfreq2D[0]**2 + kfreq2D[1]**2)
knrm = knrm.flatten()
fourier_amplitudes = fourier_amplitudes.flatten()
kbins = np.arange(0.5, npix//2+1, 1.)
kvals = 0.5 * (kbins[1:] + kbins[:-1])

Abins, _, _ = stats.binned_statistic(knrm, fourier_amplitudes,
                                      statistic = "mean",
                                      bins = kbins)
Abins *= np.pi * (kbins[1:]**2 - kbins[:-1]**2)
kvals = 2*np.pi*kvals/ ss_nm

plt.plot(kvals, Abins)
plt.loglog()
plt.xlabel('k ($nm^{-1}$)')
plt.ylabel('Mean intensity ($|F(k)|^{2}$')

### Here is a function to directly take an AFM file and turn it into a power spectrum

In [ ]:
import scipy.stats as stats
def PSD_AFM_sensor(filename, channel_name):
    ''' A function to take a raw AFM file and turn it into a PSD
    Args:
        filename (string) : filename of the AFM to read
        channel_name (string) : name of the AFM sensor to read (e.g., height or phase)
    Returns:
        kvals (numpy array) : inverse length units
        Abins (numpy array) : power spectral density (same length as kvals)
    '''
    scan = spm.Bruker(filename)
    channel1 = scan.get_channel(channel_name)
    channel_corr = channel1.correct_lines(inline=False)
    channel = channel_corr.pixels

    ss_nm, npix, px_nm = pull_params(filename,print_explicit=False)

    fourier_image = np.fft.fftn(channel)
    fourier_amplitudes = np.abs(fourier_image)**2
    kfreq = np.fft.fftfreq(npix)*npix
    kfreq2D = np.meshgrid(kfreq, kfreq)
    knrm = np.sqrt(kfreq2D[0]**2 + kfreq2D[1]**2)
    knrm = knrm.flatten()
    fourier_amplitudes = fourier_amplitudes.flatten()
    kbins = np.arange(0.5, npix//2+1, 1.)
    kvals = 0.5 * (kbins[1:] + kbins[:-1])

    Abins, _, _ = stats.binned_statistic(knrm, fourier_amplitudes,
                                         statistic = "mean",
                                         bins = kbins)
    Abins *= np.pi * (kbins[1:]**2 - kbins[:-1]**2)
    kvals = 2*np.pi*kvals/ ss_nm
    return kvals, Abins

Example usage:

In [ ]:
kvals, Abins = PSD_AFM_sensor('/content/Alahe2026/AFMdata/0p8nms/MAH27.037','Height')
plt.plot(kvals, Abins)
plt.loglog()
plt.xlabel('k ($nm^{-1}$)')
plt.ylabel('Mean intensity ($|F(k)|^{2}$')

<font color='blue'> Now, use this to plot through a filelist of the representative scans. Some of the following code may be helpful:</font>



```
for i,f in enumerate(filelist):
```
Will go through each file f, and while it does this, will have an index counter i. So, for example, while it is looking at the first file (which will be accessed using the variable f), '/content/Alahe2026/AFMdata/0p8nms/MAH27.037', i = 0. While it is looking at the second file, '/content/Alahe2026/AFMdata/0p4nms/MAH23_01.012', i = 1, and so on.


```
pcol  = iter(cm.rainbow(np.linspace(0., 1, len(filelist))))
```
with


```
plt.plot(var1, var2, color=next(pcol),label=rate_codes[i])
```
will increment a colormap for each data you plot. The pcol must be initialized before the loop starts. Rainbow is most intuitive for me. Colormaps such as viridis, cividis, magma, and Greys are best for inclusive presentations because they are perceptually uniform, meaning that if you can't perceive color, they still have saturation values that appear in order in greyscale.

The label argument will put the correct rate code in your legend, which you will add at the end of the plot with


```
plt.legend(title='Deposition rate')
```

Start this, and I will work simultaneously so you can look up for help if you are stuck.

In [ ]:
filelist = ['/content/Alahe2026/AFMdata/0p8nms/MAH27.037',
            '/content/Alahe2026/AFMdata/0p4nms/MAH23_01.012',
            '/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030',
            '/content/Alahe2026/AFMdata/0p04nms/MAH25_01.024',
            '/content/Alahe2026/AFMdata/0p008nms/MAH29.047']

rate_codes = ['0p8','0p4','0p08','0p04','0p008']

In [ ]:
# your code here

### Dependence on scan size and resolution
We're going to add in another file with a different scan size and resolution to look at PSD dependence on scan size and rate.

In [ ]:
compf = '/content/Alahe2026/AFMdata/0p08nms/MAH26_01.028'
k, A = PSD_AFM_sensor(compf,'Height')

# your code from above here, plus an added line to include this new AFM

<font color='blue'><font size="4">Use "pull_params" to investigate the origin of differences in the high-k and low-k behavior</font></font>

In [ ]:
ss_nm, npix, px_nm = pull_params('/content/Alahe2026/AFMdata/0p08nms/MAH26_01.028')
print(ss_nm, npix, px_nm)

# Task 5. Height/Phase Correlation
Here we will just plot the height and phase next to each other; if time, we will talk conceptually about ways to quantify this. It can become quite complex, and full details are out of the scope of this class. However, you can come to office hours if you're interested and want to learn more.


### Plotting Next to each other.
I'm going to do a separate cell for each rate. An example for the first file is provided; you can copy-paste to do it for each.

In [ ]:
f = filelist[0]
rate = rate_codes[0]
fig, ax = plt.subplots(1,2,figsize=(10,5))
scan = spm.Bruker(f)

channel1 = scan.get_channel('Height')
channel_corr = channel1.correct_lines(inline=False)
ht = channel_corr.pixels
channel1 = scan.get_channel('Phase')
channel_corr = channel1.correct_lines(inline=False)
phz = channel_corr.pixels
ss_nm, nlines, px_nm = pull_params(f,print_explicit=False)
cb = ax[0].imshow(ht)
ax[0].set_title(f'Height (R_dep = {rate} nm/s)')
fig.colorbar(cb, ax=ax[0], location='bottom',label='Height (nm)')


cb = ax[1].imshow(phz)
ax[1].set_title(f'Phase (R_dep = {rate} nm/s)')
fig.colorbar(cb, ax=ax[1], location='bottom',label='Phase Angle (deg)')

for i in range(2):
  scalebar = AnchoredSizeBar(ax[i].transData,
                            round(500 / px_nm), '500 nm', 'lower center',
                            pad=0.1,
                            color='white',
                            frameon=False,
                            size_vertical=3)
  ax[i].add_artist(scalebar)

I'm also going to zoom in on this so you can look a little more closely.

In [ ]:
f = filelist[0]
rate = rate_codes[0]
fig, ax = plt.subplots(1,2,figsize=(10,5))
scan = spm.Bruker(f)

channel1 = scan.get_channel('Height')
channel_corr = channel1.correct_lines(inline=False)
ht = channel_corr.pixels
channel1 = scan.get_channel('Phase')
channel_corr = channel1.correct_lines(inline=False)
phz = channel_corr.pixels
ss_nm, nlines, px_nm = pull_params(f,print_explicit=False)
cb = ax[0].imshow(ht)
ax[0].set_title(f'Height (R_dep = {rate} nm/s)')
fig.colorbar(cb, ax=ax[0], location='bottom',label='Height (nm)')


cb = ax[1].imshow(phz)
ax[1].set_title(f'Phase (R_dep = {rate} nm/s)')
fig.colorbar(cb, ax=ax[1], location='bottom',label='Phase Angle (deg)')

for i in range(2):
  ax[i].set_xlim(200,400)
  ax[i].set_ylim(200,400)

In [ ]:
# your code here for repeats

Qualitatively, what do you see?

### Linecuts.
To decrease the amount of dimensions we need to visually process over, we can take linecuts out of the image and compare on a single axis.

In [ ]:
f = filelist[0]
scan = spm.Bruker(f)

channel1 = scan.get_channel('Height')
channel_corr = channel1.correct_lines(inline=False)
ht = channel_corr.pixels
channel1 = scan.get_channel('Phase')
channel_corr = channel1.correct_lines(inline=False)
phz = channel_corr.pixels

fig, ax = plt.subplots(1,3,figsize=(15,5))
cb = ax[0].imshow(ht)
ax[0].set_title('Height')
fig.colorbar(cb, ax=ax[0], location='bottom',label='Height (nm)')
cb = ax[1].imshow(phz)
ax[1].set_title('Phase')
fig.colorbar(cb, ax=ax[1], location='bottom',label='Phase')
ax[0].axhline(250,color='red'); ax[1].axhline(250,color='orange')
ax[2].plot(ht[250,:],color='red',label='height');
ax[2].plot(phz[250,:],color='orange',label='phase')
ax[2].set_xlabel('Pixel')
ax[2].set_ylabel('Height/Phase')
ax[2].set_xlim(400,600)
ax[2].legend()

### Correlation plots.
We can also plot the height pixels vs. the phase pixels and see how correlated they are with a Pearson correlation coefficient.

In [ ]:
htf = ht.flatten()
phzf = phz.flatten()

plt.plot(htf[:5000],phzf[:5000],'o') # only plotting 5000 points because full million+ points kills the runtime

from scipy.stats import pearsonr as pr
res = pr(htf,phzf) # we can still calculate the pearson coefficient with the full set of data
print(res)
plt.xlabel('Height (nm)')
plt.ylabel('Phase angle (degrees)')

# Preview to Image processing: Binarizing

In [ ]:
plt.imshow(ht > 0)
plt.colorbar()

# Bonus: Using `glob` to automatically go through every image in each folder.
`glob` matches pathnames to automatically find different files.

Load all the files and take a look.

In [ ]:
import glob

In [ ]:
for f in glob.glob('/content/Alahe2026/AFMdata/0p08nms/M*'):
  print(f)

In [ ]:
channel_name = 'Height'
fig, ax = plt.subplots(1,6,figsize=(20,3))
for i,f in enumerate(glob.glob('/content/Alahe2026/AFMdata/0p08nms/M*')):
  scan = spm.Bruker(f)
  channel1 = scan.get_channel(channel_name)
  channel_corr = channel1.correct_lines(inline=False)
  channel = channel_corr.pixels
  ax[i].imshow(channel)

In [ ]:
# pull metadata
scan = spm.Bruker('/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030')

In [ ]:
pp = pull_params('/content/Alahe2026/AFMdata/0p08nms/MAH26_01.030',print_explicit=True)

In [ ]:
pp

In [ ]:
rate_codes = ['0p8','0p4','0p08','0p04','0p008']
rs_floats = np.asarray([0.8, 0.4, 0.08, 0.04, 0.008])
Rq_asr = np.zeros(5); Ra_asr = np.zeros(5); Rp_asr = np.zeros(5); Rv_asr = np.zeros(5); RT_asr = np.zeros(5)
i = 0
for r in rate_codes:
  scan_sizes = []; R_qs = []; R_as = []; R_ps = []; R_vs = []; R_Ts = []
  for f in glob.glob(f'/content/Alahe2026/AFMdata/{r}nms/M*'):
    scan = spm.Bruker(f)
    channel1 = scan.get_channel(channel_name)
    channel_corr = channel1.correct_lines(inline=False)
    channel = channel_corr.pixels
    ss_nm, nlines, px_nm = pull_params(f,print_explicit=False)
    R_q, R_a, R_p, R_v, R_T = calc_Rmeasures(channel)
    scan_sizes.append(ss_nm)
    R_qs.append(R_q)
    R_as.append(R_a)
    R_ps.append(R_p)
    R_vs.append(R_v)
    R_Ts.append(R_T)
  R_qs = np.asarray(R_qs)
  R_as = np.asarray(R_as)
  R_ps = np.asarray(R_ps)
  R_vs = np.asarray(R_vs)
  R_Ts = np.asarray(R_Ts)
  Rq_asr[i] = np.mean(R_qs)
  Ra_asr[i] = np.mean(R_as)
  Rp_asr[i] = np.mean(R_ps)
  Rv_asr[i] = np.mean(R_vs)
  RT_asr[i] = np.mean(R_Ts)
  i += 1

In [ ]:
plt.plot(rs_floats, Rq_asr,'o')
plt.semilogx()